# Python Foundations for Computer Vision

> **Beginner · Foundation**


## Why this matters

Vision code is still Python code. Clear functions, paths, and error handling make every later image pipeline easier to debug and reuse.

**Where it appears:** Dataset preparation, batch processing, command-line utilities, and dependable experiment scripts.


## Learning Objectives

- Write reusable functions instead of one-off script code
- Use pathlib for file handling, which the whole series relies on
- Practice the list/dict comprehension patterns used later for batch image processing


## Prerequisites

None. If you already write small Python functions comfortably, skim this notebook.

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`def`, `pathlib.Path`, exceptions, iterators and generators

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Python Refresher

Every notebook after this one manipulates *collections* of images (frames, files,
regions of interest). The Python you need is therefore centered on **functions**
(so image operations become reusable), **pathlib** (so file handling is
platform-independent), and **comprehensions** (so batch operations stay
readable). This notebook is deliberately CV-flavoured rather than a generic
Python tutorial.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Functions over scripts

A pipeline built from small, named functions is testable and reusable. Compare a flat script style to a functional style for the same task: scanning a directory to build a list of real image file records.


In [ ]:
def build_dataset(data_dir: Path) -> list[dict]:
    """Scan a directory for images and build a metadata record for each."""
    records = []
    # Sort to ensure consistent order across platforms
    for filepath in sorted(data_dir.glob("*.jpg")):
        img = cv2.imread(str(filepath))
        if img is not None:
            h, w = img.shape[:2]
            # Infer label from filename, e.g. "cat_01.jpg" -> "cat"
            label = filepath.stem.split("_")[0]
            records.append({"filename": filepath.name, "size": (w, h), "label": label})
    return records


data_dir = Path.cwd().parent / "data" / "real_images"
dataset = build_dataset(data_dir)
for record in dataset:
    print(record)

### 2. pathlib for file handling

`pathlib.Path` is used throughout this series instead of string concatenation or `os.path`. It is safer, more readable, and cross-platform.


In [ ]:
from pathlib import Path


def project_paths(root: str = ".") -> dict[str, Path]:
    """Return a standard set of project sub-directories as Path objects,
    creating them if they don't exist. Used by later notebooks that save
    intermediate results (e.g. thresholded images, detection crops)."""
    root = Path(root)
    layout = {
        "root": root,
        "data": root / "data",
        "outputs": root / "outputs",
        "models": root / "models",
    }
    for key, path in layout.items():
        if key != "root":
            path.mkdir(parents=True, exist_ok=True)
    return layout


paths = project_paths("demo_project")
for name, path in paths.items():
    print(f"{name:8s} -> {path}  (exists={path.exists()})")

### 3. Error handling for I/O

Image loading fails silently in OpenCV (`cv2.imread` returns `None` instead of raising). Wrapping loads in a validating function catches this early -- this exact pattern reappears as `safe_imread` in `cv_utils.py`.


In [ ]:
def load_or_raise(path: Path, loader) -> object:
    """Generic 'load or raise a clear error' wrapper.

    `loader` is any function that returns None on failure (like cv2.imread)
    rather than raising -- we normalize that into a proper exception.
    """
    result = loader(str(path))
    if result is None:
        raise FileNotFoundError(f"Failed to load: {path}")
    return result


def fake_loader(path_str: str):
    """Stands in for cv2.imread here so the notebook runs with no external files."""
    return None if "missing" in path_str else {"path": path_str}


for name in ["photo.jpg", "missing_photo.jpg"]:
    try:
        data = load_or_raise(Path(name), fake_loader)
        print("Loaded:", data)
    except FileNotFoundError as e:
        print("Handled error:", e)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Python Refresher: Generator and Yield

For processing very large datasets without consuming too much memory, we can use Python generators. Generators yield one item at a time instead of storing everything in a list. This is highly useful when working with thousands of high-resolution images in Computer Vision!

In [ ]:
def image_record_generator(n: int, labels=("car", "truck", "bus")):
    """Yields one image record at a time to save memory."""
    for i in range(n):
        yield {
            "filename": f"frame_{i:04d}.png",
            "size": (1920, 1080),
            "label": labels[i % len(labels)],
        }


# Let's consume the generator for 3 items
gen = image_record_generator(1000)  # Could be 1 million without using extra RAM!
for _ in range(3):
    print(next(gen))

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Python Refresher
1. Rewrite `build_dataset` so it also stores a random synthetic (x, y) 'detection' point per record.
2. Add a `filter_by_label(dataset, label)` function using a list comprehension.
3. Extend `project_paths` with a `logs` directory and confirm it's created on disk.

Use the empty cell below to work through them.


#### Solutions — Python Refresher

In [ ]:
import random


# Solution 1: Add synthetic (x, y) detection point
def build_dataset_advanced(data_dir: Path) -> list[dict]:
    records = []
    for filepath in sorted(data_dir.glob("*.jpg")):
        img = cv2.imread(str(filepath))
        if img is not None:
            h, w = img.shape[:2]
            label = filepath.stem.split("_")[0]
            records.append(
                {
                    "filename": filepath.name,
                    "size": (w, h),
                    "label": label,
                    "detection": (random.randint(0, w), random.randint(0, h)),
                }
            )
    return records

In [ ]:
# Solution 2: filter_by_label using list comprehension
def filter_by_label(dataset, label):
    return [record for record in dataset if record.get("label") == label]


ds = build_dataset_advanced(data_dir)
print("Filtered by 'cat':", filter_by_label(ds, "cat"))

In [ ]:
# Solution 3: Extend project_paths with logs
def project_paths_extended(root: str = ".") -> dict[str, Path]:
    root = Path(root)
    layout = {
        "root": root,
        "data": root / "data",
        "outputs": root / "outputs",
        "models": root / "models",
        "logs": root / "logs",
    }
    for key, path in layout.items():
        if key != "root":
            path.mkdir(parents=True, exist_ok=True)
    return layout


print("Extended paths:", project_paths_extended("demo_project"))

## Summary

You can structure a small vision task as readable, testable Python rather than a fragile sequence of notebook cells.

- **Best Practices:** Keep file paths as `Path` objects, validate inputs at boundaries, and isolate reusable operations in functions.
- **Common Pitfalls:** Relying on the current working directory, swallowing exceptions, and putting all logic in one giant cell.